# Lesson 04 — Neural Networks

Every model so far has drawn a straight boundary. Lesson 02 bent one by inventing squared
features by hand, and lesson 03 showed what happens when you invent too many. A neural
network does that feature invention **automatically**, by learning the transformation
rather than being told it.

The building block is the one from lessons 01 and 02, unchanged:

$$z = w \cdot x + b \qquad a = g(z)$$

The entire idea of this lesson is to stack that block, feed each layer's output into the
next, and work out how to compute the gradient through the whole stack. That last part is
**backpropagation**, and it is just the chain rule applied carefully.

What we build:

1. why a **nonlinearity** is mandatory, proved rather than asserted,
2. the **forward pass** through a stack of layers,
3. **backpropagation**, derived layer by layer,
4. **gradient checking**, which is how you find out you got it wrong,
5. **initialisation**, and why zero is a fatal choice here but was fine before,
6. what the hidden layer actually **learns**, drawn directly.

The reference implementation is `neural_network.py`, tested by `test_neural_network.py`.

## Notation and shapes

Layers are numbered $1 \ldots L$, with $A^{[0]} = X$ the input. The row per example
convention from every previous lesson carries over unchanged.

| symbol | shape | meaning |
|---|---|---|
| $A^{[0]} = X$ | $(m, n_0)$ | the input |
| $W^{[l]}$ | $(n_{l-1}, n_l)$ | weights of layer $l$ |
| $b^{[l]}$ | $(n_l,)$ | biases of layer $l$ |
| $Z^{[l]} = A^{[l-1]}W^{[l]} + b^{[l]}$ | $(m, n_l)$ | pre-activations, or logits |
| $A^{[l]} = g^{[l]}(Z^{[l]})$ | $(m, n_l)$ | activations |

A **layer** is one linear map followed by one activation. A network written $[2, 8, 8, 1]$
has two inputs, two hidden layers of eight units, and one output.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from neural_network import (
    NeuralNetwork, gradient_check, one_hot, zscore_normalize,
    sigmoid, sigmoid_derivative, relu, relu_derivative, tanh, tanh_derivative, softmax,
)

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

## 1. Why the nonlinearity is not optional

Suppose we drop the activation and stack two purely linear layers:

$$A^{[1]} = XW^{[1]} + b^{[1]} \qquad A^{[2]} = A^{[1]}W^{[2]} + b^{[2]}$$

Substitute the first into the second and regroup:

$$A^{[2]} = \big(XW^{[1]} + b^{[1]}\big)W^{[2]} + b^{[2]}
= X\underbrace{W^{[1]}W^{[2]}}_{\text{one matrix}} + \underbrace{b^{[1]}W^{[2]} + b^{[2]}}_{\text{one vector}}$$

which is a single linear layer with weights $W^{[1]}W^{[2]}$ and bias
$b^{[1]}W^{[2]} + b^{[2]}$. **A stack of linear layers collapses to one linear layer**, no
matter how deep. A hundred layers would still only draw a straight boundary.

The activation function is what prevents the collapse. It is the only reason depth buys
anything at all.

In [ ]:
rng_demo = np.random.default_rng(0)
X_demo = rng_demo.normal(size=(6, 3))
W1, b1 = rng_demo.normal(size=(3, 5)), rng_demo.normal(size=5)
W2, b2 = rng_demo.normal(size=(5, 2)), rng_demo.normal(size=2)

two_layers = (X_demo @ W1 + b1) @ W2 + b2
collapsed = X_demo @ (W1 @ W2) + (b1 @ W2 + b2)

print("two linear layers equal one linear layer:", np.allclose(two_layers, collapsed))
print(f"largest difference: {np.max(np.abs(two_layers - collapsed)):.2e}")

## 2. XOR, the smallest problem that needs a hidden layer

Four points. The label is 1 when exactly one input is 1.

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Plot them and the difficulty is immediate: the two positive points sit on one diagonal and
the two negative points on the other. No straight line separates them. A logistic
regression, which is exactly a network with no hidden layer, tops out at three correct out
of four, and on this perfectly symmetric data gradient descent does not even find that.

In [ ]:
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])

no_hidden = NeuralNetwork([2, 1], output="sigmoid", seed=3)
no_hidden.fit(X_xor, y_xor, alpha=0.5, epochs=4000, batch_size=4)

with_hidden = NeuralNetwork([2, 2, 1], hidden_activation="tanh", output="sigmoid", seed=0)
with_hidden.fit(X_xor, y_xor, alpha=0.5, epochs=6000, batch_size=4)

print(f"{'':>12}{'no hidden layer':>20}{'one hidden layer':>20}")
print(f"{'':>12}{'(= logistic reg)':>20}{'(2 units, tanh)':>20}")
for point, label in zip(X_xor, y_xor):
    p_flat = no_hidden.predict_proba(point.reshape(1, -1))[0, 0]
    p_deep = with_hidden.predict_proba(point.reshape(1, -1))[0, 0]
    print(f"  {point} -> {label}{p_flat:>15.4f}{p_deep:>20.4f}")
print(f"{'accuracy':>12}{no_hidden.score(X_xor, y_xor):>20.2f}"
      f"{with_hidden.score(X_xor, y_xor):>20.2f}")

# what is the best any straight line could possibly do on XOR?
best_linear = 0.0
for w1 in np.linspace(-4, 4, 81):
    for w2 in np.linspace(-4, 4, 81):
        for bias in np.linspace(-4, 4, 81):
            guess = (X_xor @ np.array([w1, w2]) + bias >= 0).astype(int)
            best_linear = max(best_linear, float(np.mean(guess == y_xor)))
print(f"\nbest accuracy achievable by ANY straight line: {best_linear:.2f}")
print("Gradient descent settles at w = 0, b = 0 instead, which outputs 0.5 everywhere.")
print("That is the symmetric solution: XOR gives every linear direction an equal and")
print("opposite reason to go the other way, so all the gradients cancel exactly.")

## 3. What the hidden layer actually does

This is the part worth internalising. The hidden layer **re-draws the input space** so that
the output layer's straight line becomes sufficient. The network has not learned a curved
boundary. It has learned a change of coordinates in which a straight boundary works.

Because we used exactly two hidden units, that new space is two dimensional and can be
plotted directly. On the left are the four points as given. On the right are the same four
points at $A^{[1]}$, the hidden layer output, with the line the output layer draws there.

In [ ]:
hidden_activations = np.tanh(X_xor @ with_hidden.W[0] + with_hidden.b[0])
w_out, b_out = with_hidden.W[1].ravel(), with_hidden.b[1][0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(X_xor[y_xor == 0, 0], X_xor[y_xor == 0, 1], s=180, c="tab:blue", label="y = 0")
axes[0].scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], s=180, c="tab:red",
                marker="^", label="y = 1")
axes[0].set_xlabel("$x_1$"); axes[0].set_ylabel("$x_2$"); axes[0].legend(fontsize=8)
axes[0].set_title("Input space: no line can separate these")

axes[1].scatter(hidden_activations[y_xor == 0, 0], hidden_activations[y_xor == 0, 1],
                s=180, c="tab:blue", label="y = 0")
axes[1].scatter(hidden_activations[y_xor == 1, 0], hidden_activations[y_xor == 1, 1],
                s=180, c="tab:red", marker="^", label="y = 1")
h_line = np.array([hidden_activations[:, 0].min() - 0.4, hidden_activations[:, 0].max() + 0.4])
axes[1].plot(h_line, -(w_out[0] * h_line + b_out) / w_out[1], "k-", lw=2,
             label="the output layer's line")
axes[1].set_xlabel("$a^{[1]}_1$"); axes[1].set_ylabel("$a^{[1]}_2$"); axes[1].legend(fontsize=8)
axes[1].set_title("Hidden space: now a line works")

g1 = np.linspace(-0.4, 1.4, 220)
G1, G2 = np.meshgrid(g1, g1)
surface = with_hidden.predict_proba(np.column_stack([G1.ravel(), G2.ravel()])).reshape(G1.shape)
contour = axes[2].contourf(G1, G2, surface, levels=20, cmap="RdBu_r", alpha=0.7)
plt.colorbar(contour, ax=axes[2], label="$P(y = 1)$")
axes[2].contour(G1, G2, surface, levels=[0.5], colors="k", linewidths=2)
axes[2].scatter(X_xor[y_xor == 0, 0], X_xor[y_xor == 0, 1], s=140, c="tab:blue", edgecolors="k")
axes[2].scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], s=140, c="tab:red",
                marker="^", edgecolors="k")
axes[2].set_xlabel("$x_1$"); axes[2].set_ylabel("$x_2$")
axes[2].set_title("Back in input space, the boundary is curved")
plt.tight_layout(); plt.show()

print("hidden activations, one row per input point:")
for point, h in zip(X_xor, hidden_activations):
    print(f"  x = {point}  ->  a1 = {h.round(4)}")

Look at the middle panel, and at the printed activations underneath it. The two points
labelled **0**, which were on opposite corners of the input square, have been mapped almost
exactly on top of each other near $(0.96, -0.95)$. The two labelled 1 have been sent
elsewhere, to roughly $(1.0, 0.96)$ and $(-0.97, -1.0)$. Once one class has been collapsed
into a single cluster and the other class sits away from it, the output layer's straight
line does the rest. The curved boundary in the right panel is that same straight line,
viewed back in the original coordinates.

This is what "learning features" means, stated concretely. Lesson 02 chose $x_1^2$ and
$x_2^2$ by hand. Here the network chose $\tanh(w \cdot x + b)$ for two different $w$ and
$b$, and it chose them by gradient descent.

### An honest caveat about two hidden units

Two units is the bare minimum for XOR, and at the minimum the training is fragile. The
cost surface is no longer convex, unlike every model in lessons 01 to 03, so gradient
descent can settle into a poor local solution depending on where it starts.

In [ ]:
print("the same architecture and data, only the random seed changes")
for seed in range(6):
    net = NeuralNetwork([2, 2, 1], hidden_activation="tanh", output="sigmoid", seed=seed)
    net.fit(X_xor, y_xor, alpha=0.5, epochs=6000, batch_size=4)
    verdict = "solved" if net.score(X_xor, y_xor) == 1.0 else "stuck in a local minimum"
    print(f"  seed {seed}:  accuracy {net.score(X_xor, y_xor):.2f}   "
          f"final cost {net.history['cost'][-1]:.5f}   {verdict}")

print("\nwith four hidden units instead of two:")
scores = []
for seed in range(6):
    net = NeuralNetwork([2, 4, 1], hidden_activation="tanh", output="sigmoid", seed=seed)
    net.fit(X_xor, y_xor, alpha=0.5, epochs=6000, batch_size=4)
    scores.append(net.score(X_xor, y_xor))
print("  accuracy by seed:", [f"{v:.2f}" for v in scores])
print(f"  solved from {sum(v == 1.0 for v in scores)} of {len(scores)} starting points")

Half the seeds fail with two units. Adding two spare units fixes it, which is the practical
reason networks are built wider than strictly necessary: extra capacity makes the
optimisation easier, quite apart from what the model can represent.

This is a real change from earlier lessons. Linear and logistic regression had convex costs
with a single minimum, so the starting point never mattered. From here on it does.

## 4. The forward pass

$$A^{[0]} = X, \qquad
Z^{[l]} = A^{[l-1]}W^{[l]} + b^{[l]}, \qquad
A^{[l]} = g^{[l]}\big(Z^{[l]}\big), \qquad l = 1 \ldots L$$

That is the whole thing. Each layer takes the previous layer's activations, applies a
linear map, and applies a nonlinearity. The last layer uses a different activation chosen
to match the task, exactly as in lessons 01 and 02:

| task | output activation | loss |
|---|---|---|
| regression | identity | mean squared error |
| binary classification | sigmoid | binary cross-entropy |
| multi class classification | softmax | categorical cross-entropy |

The forward pass also **caches** every $Z^{[l]}$ and $A^{[l]}$, because the backward pass
needs them. That is why training a network takes far more memory than making predictions
with it.

In [ ]:
net_shapes = NeuralNetwork([3, 5, 4, 2], hidden_activation="relu", output="softmax", seed=0)
X_shapes = rng.normal(size=(10, 3))
A_out, cache = net_shapes.forward(X_shapes)

print("network [3, 5, 4, 2] with 10 examples\n")
print(f"{'layer':>8}{'W shape':>12}{'b shape':>10}{'Z shape':>12}{'A shape':>12}")
print(f"{'input':>8}{'':>12}{'':>10}{'':>12}{str(cache['A'][0].shape):>12}")
for layer in range(len(net_shapes.W)):
    print(f"{layer + 1:>8}{str(net_shapes.W[layer].shape):>12}"
          f"{str(net_shapes.b[layer].shape):>10}"
          f"{str(cache['Z'][layer].shape):>12}{str(cache['A'][layer + 1].shape):>12}")

print(f"\noutput rows sum to 1 because the last layer is a softmax: "
      f"{np.allclose(A_out.sum(axis=1), 1.0)}")
total = sum(W.size for W in net_shapes.W) + sum(b.size for b in net_shapes.b)
print(f"learnable parameters: {total}")

## 5. Activation functions

| name | $g(z)$ | $g'(z)$ | range |
|---|---|---|---|
| sigmoid | $1/(1+e^{-z})$ | $g(z)(1-g(z))$ | $(0, 1)$ |
| tanh | $\tanh z$ | $1 - \tanh^2 z$ | $(-1, 1)$ |
| relu | $\max(0, z)$ | $1$ if $z > 0$, else $0$ | $[0, \infty)$ |

The **rectified linear unit** is the default for hidden layers and it is almost
embarrassingly simple. Its usefulness comes from its derivative: exactly $1$ on the
positive side, rather than something that shrinks.

Strictly, relu has no derivative at $z = 0$. Any value in $[0, 1]$ is a valid subgradient
there and the convention is to use $0$. A single point of measure zero never comes up in
practice.

In [ ]:
z = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, fn, derivative in [("sigmoid", sigmoid, sigmoid_derivative),
                             ("tanh", tanh, tanh_derivative),
                             ("relu", relu, relu_derivative)]:
    axes[0].plot(z, fn(z), lw=2, label=name)
    axes[1].plot(z, derivative(z), lw=2, label=name)
axes[0].set_xlabel("$z$"); axes[0].set_ylabel("$g(z)$"); axes[0].legend(fontsize=8)
axes[0].set_title("Activation functions")
axes[1].set_xlabel("$z$"); axes[1].set_ylabel("$g'(z)$"); axes[1].legend(fontsize=8)
axes[1].set_title("Their derivatives, which is what backprop multiplies by")
plt.tight_layout(); plt.show()

print(f"largest value of the sigmoid derivative: {sigmoid_derivative(np.array([0.0]))[0]:.4f}")
print(f"largest value of the tanh derivative   : {tanh_derivative(np.array([0.0]))[0]:.4f}")
print(f"the relu derivative is exactly 1 wherever z > 0")

### The vanishing gradient problem

The right panel explains why sigmoid was abandoned for hidden layers. Its derivative peaks
at $0.25$ and is far smaller almost everywhere. Backpropagation multiplies one such factor
per layer, so after $k$ layers the gradient reaching the front of the network has been
scaled by at most $0.25^k$. For six layers that is a factor of $2.4 \times 10^{-4}$ before
anything else is taken into account.

The early layers then receive almost no gradient and barely train. This is the **vanishing
gradient** problem, and it is why deep networks were considered impractical for years.

In [ ]:
X_deep = rng.normal(size=(200, 10))
Y_deep = (rng.random((200, 1)) > 0.5).astype(float)

print("a network with six hidden layers of 16 units, gradients at initialisation")
print(f"\n{'activation':>12}" + "".join(f"{f'layer {i+1}':>11}" for i in range(7)))
for name in ("sigmoid", "tanh", "relu"):
    deep_net = NeuralNetwork([10] + [16] * 6 + [1], hidden_activation=name,
                             output="sigmoid", seed=0)
    _, deep_cache = deep_net.forward(X_deep)
    dW_deep, _ = deep_net.backward(Y_deep, deep_cache)
    magnitudes = [float(np.abs(g).mean()) for g in dW_deep]
    print(f"{name:>12}" + "".join(f"{v:>11.1e}" for v in magnitudes))

deep_sigmoid = NeuralNetwork([10] + [16] * 6 + [1], hidden_activation="sigmoid",
                            output="sigmoid", seed=0)
_, c = deep_sigmoid.forward(X_deep)
g, _ = deep_sigmoid.backward(Y_deep, c)
print(f"\nwith sigmoid the last layer's gradient is "
      f"{np.abs(g[-1]).mean() / np.abs(g[0]).mean():,.0f} times larger than the first layer's")

The sigmoid row falls off a cliff. Its first layer receives a gradient of roughly
$10^{-7}$ while the output layer gets about $10^{-2}$, a ratio of order $10^{5}$ as the
printed figure shows. In the time it takes the output layer to learn something, the first layer has
barely moved. The relu and tanh rows stay within one or two orders of magnitude across all
seven layers.

Sigmoid remains the right choice for a **binary output**, where the job is producing a
probability rather than passing a signal onward.

## 6. Backpropagation

We need $\partial J/\partial W^{[l]}$ and $\partial J/\partial b^{[l]}$ for every layer. A
weight in layer 1 affects the cost only through everything that comes after it, so the
chain rule has to be applied through the whole stack.

The trick that makes it efficient is to compute, for each layer, the gradient of the cost
with respect to that layer's **output**, and pass it backwards. Write
$dA^{[l]} = \partial J/\partial A^{[l]}$. Then for each layer, working from the last to the
first:

$$
\begin{aligned}
dZ^{[l]} &= dA^{[l]} \odot g'\big(Z^{[l]}\big) && (m, n_l) \\[4pt]
dW^{[l]} &= \tfrac{1}{m}\,\big(A^{[l-1]}\big)^\top dZ^{[l]} && (n_{l-1}, n_l) \\[4pt]
db^{[l]} &= \tfrac{1}{m}\sum_{i=1}^{m} dZ^{[l]}_{i} && (n_l,) \\[4pt]
dA^{[l-1]} &= dZ^{[l]}\big(W^{[l]}\big)^\top && (m, n_{l-1})
\end{aligned}
$$

where $\odot$ is entrywise multiplication. Four lines, and they are all you need.

Read them in order. The first converts the gradient with respect to a layer's output into
the gradient with respect to its pre-activation, by multiplying by the local derivative of
the activation. The next two extract this layer's parameter gradients. The fourth hands the
gradient back to the previous layer, and the loop repeats.

Each shape is forced. $dW^{[l]}$ must match $W^{[l]}$, and $(n_{l-1}, m) \times (m, n_l)$
is the only way to combine the available matrices to get $(n_{l-1}, n_l)$. Checking shapes
is the fastest way to catch a mistake in these formulas.

### The output layer, where the algebra collapses

Starting the recursion needs $dZ^{[L]}$ at the output. Computing $dA^{[L]}$ and then
multiplying by $g'$ would work but is both clumsy and numerically poor, because the
cross-entropy derivative contains $1/f$ and $1/(1-f)$, which explode as predictions
saturate.

Instead, fuse the loss and the output activation. For sigmoid with binary cross-entropy the
$\sigma(1-\sigma)$ factor cancels, exactly as it did in lesson 02. For softmax with
categorical cross-entropy the same cancellation happens. In both cases:

$$\boxed{\;dZ^{[L]} = \frac{1}{m}\big(A^{[L]} - Y\big)\;}$$

Prediction minus label, one more time. This is the fourth lesson in a row where that
expression has appeared, and it is the same reason each time: these are all generalised
linear models fitted by maximum likelihood, and the linear output layer sits at the end of
the network.

In [ ]:
# the fused output gradient, checked against the definition on a network with no hidden layer
check_net = NeuralNetwork([3, 1], output="sigmoid", seed=0)
X_check = rng.normal(size=(9, 3))
Y_check = (rng.random((9, 1)) > 0.5).astype(float)

A_check, cache_check = check_net.forward(X_check)
dW_check, db_check = check_net.backward(Y_check, cache_check)

expected = X_check.T @ (A_check - Y_check) / len(Y_check)
print("with no hidden layer the network is exactly logistic regression, so its gradient")
print("should equal the lesson 02 formula X.T @ (predictions - labels) / m")
print(f"  matches: {np.allclose(dW_check[0], expected)}")
print(f"  largest difference: {np.max(np.abs(dW_check[0] - expected)):.2e}")

## 7. Gradient checking, which is not optional here

Backpropagation is the first algorithm in these lessons complicated enough that a wrong
implementation still *appears* to work. A network with a subtly wrong gradient still
trains, just worse, and the symptoms look exactly like bad hyperparameters. You can lose
days to that.

The remedy is the same central difference from lesson 01, now applied to every parameter,
and summarised as a relative error:

$$\text{relative error} = \frac{\lVert \nabla_{\text{analytic}} - \nabla_{\text{numeric}} \rVert}{\lVert \nabla_{\text{analytic}} \rVert + \lVert \nabla_{\text{numeric}} \rVert}$$

Below about $10^{-7}$ the two agree. Above $10^{-3}$ there is a real bug.

It is far too slow to use during training, since it costs two full forward passes per
parameter. Run it once on a small network, then turn it off.

In [ ]:
print(f"{'hidden activation':>20}{'output':>12}{'relative error':>18}{'verdict':>12}")
for activation in ("relu", "tanh", "sigmoid"):
    for output, n_out in (("sigmoid", 1), ("softmax", 3), ("linear", 2)):
        rng_gc = np.random.default_rng(3)
        X_gc = rng_gc.normal(size=(12, 4))
        if output == "softmax":
            Y_gc = one_hot(rng_gc.integers(0, n_out, 12), n_out)
        elif output == "sigmoid":
            Y_gc = (rng_gc.random((12, 1)) > 0.5).astype(float)
        else:
            Y_gc = rng_gc.normal(size=(12, n_out))

        net_gc = NeuralNetwork([4, 5, 4, n_out], hidden_activation=activation,
                               output=output, lam=0.5, seed=1)
        error = gradient_check(net_gc, X_gc, Y_gc)
        print(f"{activation:>20}{output:>12}{error:>18.3e}"
              f"{'pass' if error < 1e-7 else 'FAIL':>12}")

Every combination of hidden activation and output passes, with the L2 penalty switched on
as well. That is the whole of the backward pass verified in one cell.

## 8. Initialisation

Lessons 01 to 03 all started from $w = 0$ without a second thought. Here that is fatal.

If every weight in a layer is zero, then every unit in that layer computes the same output,
receives the same gradient, and takes the same update. They stay identical forever, so a
layer of 64 units has the representational power of exactly one unit. This is the
**symmetry breaking** problem, and random initialisation is what solves it.

The scale matters too. Too small and the signal shrinks as it passes through layers until
nothing reaches the output. Too large and it grows until the activations saturate. Two
standard choices keep the variance roughly constant from layer to layer:

$$\text{He (for relu)}: \; \sigma = \sqrt{\frac{2}{n_{l-1}}}
\qquad
\text{Xavier (for tanh and sigmoid)}: \; \sigma = \sqrt{\frac{1}{n_{l-1}}}$$

Biases can safely start at zero, because the weights have already broken the symmetry.

In [ ]:
X_init = rng.normal(size=(500, 64))
depths = 8

print("standard deviation of the activations, layer by layer, at initialisation")
print(f"\n{'scale':>26}" + "".join(f"{f'L{i+1}':>9}" for i in range(depths)))
for label, scale_fn in [("too small (0.01)", lambda n: 0.01),
                        ("He, sqrt(2/n)", lambda n: np.sqrt(2.0 / n)),
                        ("too large (1.0)", lambda n: 1.0)]:
    A = X_init
    spreads = []
    seed_rng = np.random.default_rng(0)
    for _ in range(depths):
        W = seed_rng.normal(0.0, scale_fn(A.shape[1]), size=(A.shape[1], 64))
        A = relu(A @ W)
        spreads.append(float(A.std()))
    print(f"{label:>26}" + "".join(f"{v:>9.2e}" for v in spreads))

print("\nToo small and the signal dies out. Too large and it explodes. He initialisation")
print("holds it roughly steady, which is exactly what it was designed to do.")

In [ ]:
# symmetry breaking, shown directly
X_sym = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_sym = np.array([0, 1, 1, 0])

zero_net = NeuralNetwork([2, 4, 1], hidden_activation="tanh", output="sigmoid", seed=0)
for layer in range(len(zero_net.W)):
    zero_net.W[layer] = np.zeros_like(zero_net.W[layer])      # deliberately broken
zero_net.fit(X_sym, y_sym, alpha=0.5, epochs=3000, batch_size=4)

hidden_after = np.tanh(X_sym @ zero_net.W[0] + zero_net.b[0])
print("after training a network whose weights all started at exactly zero:")
print(f"  hidden layer weights, column by column:\n{zero_net.W[0].round(6)}")
print(f"\n  all four hidden units identical: "
      f"{np.allclose(hidden_after, hidden_after[:, [0]])}")
print(f"  accuracy on XOR: {zero_net.score(X_sym, y_sym):.2f}  (a coin flip)")

Every column of the hidden weight matrix is identical, so all four units compute the same
number, and the network is stuck at chance on a problem it solves easily when initialised
randomly.

## 9. A harder problem: two interleaving spirals

XOR is a toy. Two spiral arms wound around each other need a genuinely curved boundary that
no hand-picked polynomial would produce easily.

In [ ]:
def make_spirals(n_per_class, noise=0.06, seed=0):
    r = np.random.default_rng(seed)
    t = np.sqrt(r.uniform(0.05, 1, n_per_class)) * 2.6 * np.pi
    points, labels = [], []
    for k in (0, 1):
        angle = t + k * np.pi
        arm = np.column_stack([t * np.cos(angle), t * np.sin(angle)]) / 9.0
        points.append(arm + r.normal(0, noise, arm.shape))
        labels.append(np.full(n_per_class, k))
    return np.vstack(points), np.concatenate(labels)


X_spiral, y_spiral = make_spirals(300, seed=1)
print("X shape", X_spiral.shape, " class counts", np.bincount(y_spiral))

architectures = [("logistic regression", [2, 1], "relu"),
                 ("one hidden layer, 4 units", [2, 4, 1], "relu"),
                 ("one hidden layer, 16 units", [2, 16, 1], "relu"),
                 ("one hidden layer, 256 units", [2, 256, 1], "relu"),
                 ("two hidden layers, 16 each", [2, 16, 16, 1], "relu"),
                 ("three hidden layers, 16 each", [2, 16, 16, 16, 1], "relu")]

trained = {}
print(f"\n{'architecture':>32}{'parameters':>12}{'accuracy':>11}")
for name, sizes, activation in architectures:
    net = NeuralNetwork(sizes, hidden_activation=activation, output="sigmoid", seed=1)
    net.fit(X_spiral, y_spiral, alpha=0.3, epochs=400, batch_size=32)
    trained[name] = net
    n_params = sum(W.size for W in net.W) + sum(b.size for b in net.b)
    print(f"{name:>32}{n_params:>12}{net.score(X_spiral, y_spiral):>11.4f}")

In [ ]:
show = ["logistic regression", "one hidden layer, 4 units", "one hidden layer, 16 units",
        "one hidden layer, 256 units", "two hidden layers, 16 each",
        "three hidden layers, 16 each"]

span = np.linspace(-1.15, 1.15, 220)
S1, S2 = np.meshgrid(span, span)
flat_grid = np.column_stack([S1.ravel(), S2.ravel()])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, name in zip(axes.ravel(), show):
    net = trained[name]
    surface = net.predict_proba(flat_grid).reshape(S1.shape)
    ax.contourf(S1, S2, surface, levels=20, cmap="RdBu_r", alpha=0.6)
    ax.contour(S1, S2, surface, levels=[0.5], colors="k", linewidths=1.6)
    ax.scatter(X_spiral[y_spiral == 0, 0], X_spiral[y_spiral == 0, 1], s=6, c="tab:blue")
    ax.scatter(X_spiral[y_spiral == 1, 0], X_spiral[y_spiral == 1, 1], s=6, c="tab:red")
    n_params = sum(W.size for W in net.W) + sum(b.size for b in net.b)
    ax.set_title(f"{name}\n{n_params} parameters, accuracy "
                 f"{net.score(X_spiral, y_spiral):.3f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

### Depth against width

Compare two of those rows. One hidden layer of 256 units and two hidden layers of 16 units
both reach perfect accuracy, but the deep one does it with roughly a third of the
parameters. That is the practical argument for depth: layers **compose**, so each one
refines the representation the previous one produced, and composition reaches complicated
functions more economically than raw width does.

The universal approximation theorem says a single hidden layer can approximate any
continuous function to any accuracy given enough units. It is true and it is not the point.
It says nothing about *how many* units, and depth is usually far cheaper.

Notice also that 4 hidden units is not enough here, at 81 percent. Capacity is a real
constraint, and the failure looks like underfitting, exactly as in lesson 03.

## 10. Overfitting has not gone away

A network with enough parameters will memorise, and everything from lesson 03 still
applies. The L2 penalty attaches to the weight matrices in exactly the same way, with the
biases again exempt.

In [ ]:
X_small, y_small = make_spirals(120, noise=0.15, seed=4)
X_held, y_held = make_spirals(400, noise=0.15, seed=9)

print(f"a [2, 64, 64, 1] network, {sum(W.size for W in NeuralNetwork([2,64,64,1]).W) + 129} "
      f"parameters, trained on only {len(y_small)} points")
print(f"evaluated on {len(y_held)} held out points\n")
print(f"{'lambda':>8}{'train':>10}{'held out':>11}{'gap':>9}{'sum of squared weights':>25}")
for lam in (0.0, 0.01, 0.1, 0.3, 1.0, 3.0):
    net = NeuralNetwork([2, 64, 64, 1], hidden_activation="relu", output="sigmoid",
                        lam=lam, seed=2)
    net.fit(X_small, y_small, alpha=0.3, epochs=800, batch_size=16)
    train_acc = net.score(X_small, y_small)
    held_acc = net.score(X_held, y_held)
    weight_size = sum(float(np.sum(W ** 2)) for W in net.W)
    print(f"{lam:>8}{train_acc:>10.4f}{held_acc:>11.4f}{train_acc - held_acc:>9.4f}"
          f"{weight_size:>25.2f}")

The pattern is the one from lesson 03. Training accuracy is highest with no penalty and
falls as $\lambda$ grows. The total weight size collapses by more than two orders of
magnitude. Held out accuracy peaks away from zero, and the **gap** column, which is the
generalisation gap, narrows before the model starts underfitting.

Being straight about the size of the effect: the held out gain here is a couple of
percentage points, not a transformation. The unambiguous signals are the weight size and
the narrowing gap. Regularisation on this problem is doing real work, but a network with
64 by 64 hidden units trained on 240 points is short of data more than it is short of
constraint, and lesson 03 exercise 5 already showed which of those a learning curve tells
you to fix.

Techniques specific to networks, which this lesson does not implement, include dropout
(randomly zeroing activations during training), batch normalisation, and data augmentation.
Early stopping from exercise 2 of lesson 03 is used almost universally.

## 11. Multiple classes with softmax

For $K$ mutually exclusive classes, the output layer has $K$ units and a softmax:

$$\operatorname{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

which turns a row of $K$ logits into a probability distribution. It is the direct
generalisation of the sigmoid, and for $K = 2$ it reduces to one. The loss is categorical
cross-entropy, $-\sum_k y_k \log f_k$, and with labels one-hot encoded only the true
class's term survives.

Softmax needs the same numerical care as the sigmoid did. Subtracting the row maximum
before exponentiating changes nothing mathematically, since the factor cancels between
numerator and denominator, but it guarantees the largest exponent is $e^0 = 1$ so nothing
overflows.

In [ ]:
print("softmax on huge logits, where a naive implementation overflows:")
huge = np.array([[1000.0, 1001.0, 999.0]])
print(f"  stable version : {softmax(huge)}")
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(huge) / np.exp(huge).sum(axis=1, keepdims=True)
print(f"  naive version  : {naive}   (inf divided by inf)")
print(f"\n  shifting every logit by a constant leaves the answer unchanged: "
      f"{np.allclose(softmax(huge), softmax(np.array([[1.0, 2.0, 0.0]])))}")

In [ ]:
centres = [[-3.0, 0.5], [3.0, 0.0], [0.0, 3.5], [0.5, -3.0]]
X_multi = np.vstack([rng.normal(c, 0.75, size=(120, 2)) for c in centres])
y_multi = np.repeat(np.arange(len(centres)), 120)

multi_net = NeuralNetwork([2, 24, 24, 4], hidden_activation="relu", output="softmax",
                          lam=0.01, seed=5)
multi_net.fit(X_multi, one_hot(y_multi, 4), alpha=0.2, epochs=400, batch_size=32)
print(f"four class accuracy: {multi_net.score(X_multi, y_multi):.4f}")

probabilities = multi_net.predict_proba(X_multi[:3])
print("\npredicted distributions for the first three points:")
for row, label in zip(probabilities, y_multi[:3]):
    print(f"  true class {label}: {row.round(4)}  sums to {row.sum():.6f}")

span_m = np.linspace(X_multi[:, 0].min() - 1, X_multi[:, 0].max() + 1, 220)
span_n = np.linspace(X_multi[:, 1].min() - 1, X_multi[:, 1].max() + 1, 220)
M1, M2 = np.meshgrid(span_m, span_n)
predicted = multi_net.predict(np.column_stack([M1.ravel(), M2.ravel()])).reshape(M1.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].contourf(M1, M2, predicted, levels=[-0.5, 0.5, 1.5, 2.5, 3.5], cmap="Set2", alpha=0.4)
for k in range(4):
    axes[0].scatter(X_multi[y_multi == k, 0], X_multi[y_multi == k, 1], s=8, label=f"class {k}")
axes[0].legend(fontsize=8); axes[0].set_xlabel("$x_1$"); axes[0].set_ylabel("$x_2$")
axes[0].set_title("Four classes, one softmax output layer")

axes[1].plot(multi_net.history["epoch"], multi_net.history["cost"])
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("cost")
axes[1].set_title("Categorical cross-entropy during training")
plt.tight_layout(); plt.show()

## Exercises

1. **Break backpropagation on purpose.** Introduce one bug at a time into `backward`:
   forget the division by $m$, use $W$ instead of $W^\top$ when computing `dA_prev`, apply
   the activation derivative to $A$ instead of $Z$. For each, report what `gradient_check`
   returns and whether the network still trains. Which bug is hardest to notice from the
   learning curve alone?

2. **Draw the learned features.** Train a network with 3 hidden units on XOR and plot the
   four data points in the three dimensional hidden space, together with the plane the
   output layer draws. Then repeat with `relu` instead of `tanh` and describe how the
   picture changes.

3. **Dead units.** A relu unit whose input is negative for every training example has zero
   gradient forever and can never recover. Train a network on the spirals with a large
   learning rate, then count how many hidden units output zero for all inputs. Repeat with
   a small learning rate and compare. Explain the connection to the learning rate.

4. **Regression with a network.** Use `output="linear"` to fit
   $y = \sin(3x) + 0.3x^2$ on $[-3, 3]$ with noise. Compare against the degree 12
   polynomial ridge model from lesson 03 on held out data. Which extrapolates better
   outside the training range, and why should you be suspicious of both?

5. **Depth against width at fixed budget.** For a parameter budget of roughly 1000, build
   the widest single hidden layer network that fits, and a network of three or four hidden
   layers with the same total. Train both on the spirals ten times with different seeds and
   compare mean accuracy and its spread. Does depth win, and is the difference bigger than
   the seed to seed variation?

## Where this leaves you

Four lessons, and the loop has not changed once:

$$\text{model} \;\longrightarrow\; \text{cost} \;\longrightarrow\; \text{gradient} \;\longrightarrow\; \text{gradient descent}$$

Linear regression fixed the model as a line and the cost as squared error. Logistic
regression wrapped the line in a sigmoid and switched to cross-entropy. Regularisation
added a penalty to the cost. Neural networks stacked the model and used the chain rule to
get the gradient. The optimiser has been the same code every time, and the expression
$\text{prediction} - \text{label}$ has turned up in all four.

What a modern framework adds on top of this is mostly automation and scale: automatic
differentiation so nobody hand derives a backward pass, better optimisers such as Adam,
architectures suited to particular data such as convolutions for images and attention for
sequences, and GPU kernels. None of it changes the loop.